In [11]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import kagglehub
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

# These two are separate libraries, not part of sklearn - need pip install if not already available
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# MLflow to keep record of models performance
import mlflow
import mlflow.sklearn


In [12]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load



# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md


# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv


In [14]:
df = pd.read_csv("/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors='coerce')
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# pd.set_option("display.max_columns",None)
# print (df.head(10))
# print (df.info())

In [15]:
def feature_engineering(x_train , x_test):
    encoder = OneHotEncoder(sparse_output = False , handle_unknown ="ignore").set_output(transform="pandas")
    x_train_encoded = encoder.fit_transform(x_train)
    x_test_encoded = encoder.transform(x_test)
    return x_train_encoded , x_test_encoded

In [16]:
def feature_scaling(x_train,x_test):
    std = StandardScaler()
    x_train_scaled = pd.DataFrame(std.fit_transform(x_train),columns=x_train.columns,index=x_train.index)
    x_test_scaled = pd.DataFrame(std.transform(x_test),columns=x_test.columns,index=x_test.index)

    return x_train_scaled,x_test_scaled


In [17]:
yes_no_cols = [col for col in df.columns if set(df[col].dropna().unique()) == {'Yes', 'No'}]
print(yes_no_cols)

for col in yes_no_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']


In [18]:
mlflow.set_tracking_uri('file:/kaggle/working/mlruns')
mlflow.set_experiment('customer_churn_prediction')

def Classification(model,x_train,y_train,x_test,y_test):
    for name,model in model.items():
        model.fit(x_train,y_train)
        pred = model.predict(x_test)
        train_score = model.score(x_train,y_train)
    
        train_acc = model.score(x_train, y_train)
        test_acc = accuracy_score(y_test, pred)
        precision = precision_score(y_test, pred)
        recall = recall_score(y_test, pred)
        f1 = f1_score(y_test, pred)

        mlflow.log_param('model_type', name)
        mlflow.log_metric('train_accuracy', train_acc)
        mlflow.log_metric('test_accuracy', test_acc)
        mlflow.log_metric('precision', precision)
        mlflow.log_metric('recall', recall)
        mlflow.log_metric('f1_score', f1)
        mlflow.sklearn.log_model(model, 'model')
        
        print(f"--- {name} ---")
        print(f"Train Accuracy : {train_acc:.4f}")
        print(f"Test Accuracy  : {test_acc:.4f}")
        print(f"Precision      : {precision:.4f}")
        print(f"Recall         : {recall:.4f}")
        print(f"F1 Score       : {f1:.4f}")
        print(f"Confusion Matrix:\n{confusion_matrix(y_test, pred)}\n")


MlflowException: The filesystem tracking backend (e.g., './mlruns') is in maintenance mode and will not receive further updates. Please migrate to a database backend (e.g., 'sqlite:///mlflow.db') to access the latest MLflow features. The `mlflow migrate-filestore` tool migrates your existing data losslessly. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance. If the filesystem backend is required for your workflow, set `MLFLOW_ALLOW_FILE_STORE=true` to opt out of this exception.

In [17]:
   
x_train,x_test, y_train , y_test = train_test_split (df.drop(columns=["Churn","customerID"]),df["Churn"],random_state=42)

    
x_train_numeric = x_train.select_dtypes(include=['int64','float64'])
x_train_variable = x_train.select_dtypes(include=['object'])

x_test_numeric = x_test.select_dtypes(include=['int64','float64'])
x_test_variable = x_test.select_dtypes(include=['object'])

x_train_variable_encoded,x_test_variable_encoded = feature_engineering(x_train_variable,x_test_variable)

x_train_encoded = pd.concat([x_train_variable_encoded,x_train_numeric],axis=1)
x_test_encoded = pd.concat([x_test_variable_encoded,x_test_numeric],axis=1)

print(x_train_encoded.shape)
# print(df.corr(numeric_only=True)['Churn'].sort_values(ascending=False))

x_train_scaled,x_test_scaled = feature_scaling(x_train_encoded,x_test_encoded)
print(x_train_scaled.isnull().sum())


(5282, 41)
gender_Female                              0
gender_Male                                0
MultipleLines_No                           0
MultipleLines_No phone service             0
MultipleLines_Yes                          0
InternetService_DSL                        0
InternetService_Fiber optic                0
InternetService_No                         0
OnlineSecurity_No                          0
OnlineSecurity_No internet service         0
OnlineSecurity_Yes                         0
OnlineBackup_No                            0
OnlineBackup_No internet service           0
OnlineBackup_Yes                           0
DeviceProtection_No                        0
DeviceProtection_No internet service       0
DeviceProtection_Yes                       0
TechSupport_No                             0
TechSupport_No internet service            0
TechSupport_Yes                            0
StreamingTV_No                             0
StreamingTV_No internet service            0

In [18]:
model = {
    "Logistic Regression":LogisticRegression()}
Classification(model,x_train_scaled,y_train,x_test_scaled,y_test)

--- Logistic Regression ---
Train Accuracy : 0.8037
Test Accuracy  : 0.8132
Precision      : 0.6847
Recall         : 0.5804
F1 Score       : 0.6282
Confusion Matrix:
[[1154  128]
 [ 201  278]]



In [9]:

print(mlflow.__version__)

3.16.0
